In [26]:
import sys
import os
import pandas as pd
import numpy as np
from IPython.display import display
from typing import Optional, Dict, Any

import warnings
# Ignore *all* warnings
warnings.filterwarnings("ignore")

# Update the visualisation of the notebook
from typing import Optional, Dict, Any
from rich.console import Console
from rich.table import Table
from rich.prompt import Prompt
from rich.theme import Theme


In [27]:
 # !pip3 install rich -> installation needed

In [28]:
console = Console()

def data_overview(
    data: pd.DataFrame,
    columns: Optional[List[str]] = None,
    return_data: bool = False,
    max_describe_cols: int = 12,
    menu_mode: bool = True
) -> Optional[Dict[str, Any]]:
    """
    Enhanced interactive preview of a pandas DataFrame, with color-coded comprehensive display.
    """
    # Input type check
    if not isinstance(data, pd.DataFrame):
        raise TypeError(f"Expected pandas DataFrame, got {type(data)}")

    # Empty DataFrame case
    if data.empty:
        console.print("[bold red]Empty DataFrame — nothing to display[/]")
        if return_data:
            return {
                "missing": pd.DataFrame(),
                "unique": pd.DataFrame(),
                "describe_numeric": pd.DataFrame(),
                "describe_object": pd.DataFrame(),
                "info": pd.DataFrame(),
                "dtypes": pd.Series(dtype=object)
            }
        return None

    # Work on a copy (subset columns if provided)
    df = data[columns].copy() if columns else data.copy()

    # Dimensions and total memory
    num_rows, num_cols = df.shape
    mem_total = df.memory_usage(deep=True).sum()
    mem_mb = round(mem_total / 1024**2, 2)

    # ----------------------- Column statistics -----------------------
    def _col_stats(df):
        stats = []
        for col in df.columns:
            s = df[col]
            miss_ct = int(s.isna().sum())
            miss_pct = round((miss_ct / len(s)) * 100, 2)
            uniq_ct = int(s.nunique(dropna=True))
            uniq_pct = round((uniq_ct / len(s)) * 100, 2)
            dtype = str(s.dtype)
            top_vals = s.value_counts().head(3).to_dict()
            stats.append({
                "column": col,
                "missing_count": miss_ct,
                "missing_%": miss_pct,
                "unique_count": uniq_ct,
                "unique_%": uniq_pct,
                "dtype": dtype,
                "top_3_values": top_vals
            })
        return pd.DataFrame(stats).set_index("column")

    stats_df = _col_stats(df)
    missing_df = stats_df[["missing_count", "missing_%"]].sort_values(by="missing_count", ascending=False)
    unique_df = stats_df[["unique_count", "unique_%"]].sort_values(by="unique_count", ascending=False)
    # Descriptive stats (pandas describe) for numeric and object columns
    numeric_desc = df.select_dtypes(include=[np.number]).describe().T.head(max_describe_cols)
    object_desc = df.select_dtypes(include=['object', 'category']).describe().T.head(max_describe_cols)
    dtypes_ser = df.dtypes

    # ----------------------- Info table -----------------------
    non_null = df.notna().sum()
    mem_usage = df.memory_usage(deep=True)
    # If 'Index' appears (depending on index), drop it
    if 'Index' in mem_usage.index:
        mem_usage = mem_usage.drop('Index')
    info_table = Table(title="DataFrame Info", show_lines=True)
    info_table.add_column("Column", style="cyan")
    info_table.add_column("Non-null", justify="right")
    info_table.add_column("Type", style="yellow")
    info_table.add_column("Memory (MB)", justify="right")
    for col in df.columns:
        mem_mb_col = round(mem_usage.get(col, 0) / 1024**2, 2)
        info_table.add_row(col, str(non_null[col]), str(dtypes_ser[col]), str(mem_mb_col))

    # ----------------------- Duplicates table -----------------------
    dup_count = int(df.duplicated().sum())
    dup_table = Table(title="Duplicates", show_lines=True)
    dup_table.add_column("Total rows", justify="right", style="cyan")
    dup_table.add_column("Duplicated rows", justify="right", style="red")
    dup_table.add_row(str(num_rows), str(dup_count))

    # ----------------------- Interactive menu -----------------------
    if not menu_mode:
        return stats_df if return_data else None

    while True:
        menu = Table(title="Data Overview Menu", show_header=True, header_style="bold magenta")
        menu.add_column("No.", style="cyan", justify="center")
        menu.add_column("Option", style="green")
        menu.add_column("Description", style="yellow")
        options = {
            "1": "Missing values",
            "2": "Unique values",
            "3": "DataFrame head (preview)",
            "4": "Descriptive statistics",
            "5": "Info (non-null, type, memory)",
            "6": "Duplicates",
            "7": "All",
            "8": "Quit"
        }
        for k, v in options.items():
            menu.add_row(k, v, f"Show {v.lower()}")
        console.print(menu)
        choice = console.input("[bold white]Enter choice(s) separated by comma: [/]")
        choices = [c.strip() for c in choice.split(",") if c.strip()]

        # Quit option
        if "8" in choices:
            console.print("[bold red]Exiting menu...[/]")
            break

        # Missing values
        if "1" in choices:
            t = Table(title="Missing Values", show_lines=True)
            t.add_column("Column", style="cyan")
            t.add_column("Missing", justify="right", style="white")
            t.add_column("%", justify="right", style="white")
            for idx, row in missing_df.iterrows():
                t.add_row(idx, str(int(row["missing_count"])), str(row["missing_%"]))
            console.print(t)

        # Unique values
        if "2" in choices:
            t = Table(title="Unique Values", show_lines=True)
            t.add_column("Column", style="cyan")
            t.add_column("Unique", justify="right", style="green")
            t.add_column("%", justify="right", style="green")
            for idx, row in unique_df.iterrows():
                t.add_row(idx, str(int(row["unique_count"])), str(row["unique_%"]))
            console.print(t)

        # DataFrame head (preview)
        if "3" in choices:
            console.print("[bold green]DataFrame head (preview):[/]")
            head = df.head()
            t = Table(show_header=True, header_style="bold cyan")
            t.add_column("Index", style="dim", justify="right")
            for col in head.columns:
                t.add_column(col, style="magenta")
            for idx, row in head.iterrows():
                t.add_row(str(idx), *[str(v) for v in row.tolist()])
            console.print(t)

        # Descriptive statistics
        if "4" in choices:
            console.print("[bold green]Numeric descriptive statistics:[/]")
            console.print(numeric_desc)
            console.print("[bold green]Categorical descriptive statistics:[/]")
            console.print(object_desc)

        # DataFrame Info
        if "5" in choices:
            console.print("[bold green]DataFrame Info:[/]")
            console.print(info_table)

        # Duplicates
        if "6" in choices:
            console.print("[bold green]Duplicates in DataFrame:[/]")
            console.print(dup_table)
            if dup_count > 0:
                console.print("[bold yellow]Example duplicated rows:[/]")
                console.print(df[df.duplicated()])

        # Show all information
        if "7" in choices:
            console.print("[bold green]All information:[/]")
            console.print(f"[bold green]Shape:[/bold green] {num_rows} rows, {num_cols} columns (Memory: {mem_mb} MB)")
            console.print("[bold green]Missing values:[/bold green]")
            console.print(missing_df)
            console.print("[bold green]Unique values:[/bold green]")
            console.print(unique_df)
            console.print("[bold green]DataFrame head:[/bold green]")
            console.print(df.head())
            console.print("[bold green]Numeric descriptive statistics:[/bold green]")
            console.print(numeric_desc)
            console.print("[bold green]Categorical descriptive statistics:[/bold green]")
            console.print(object_desc)
            console.print("[bold green]DataFrame Info (non-null, type, memory):[/bold green]")
            console.print(info_table)
            console.print("[bold green]Duplicates:[/bold green]")
            console.print(dup_table)

        console.print("[bold blue]Done. Choose another option or 8 to quit[/]\n")

    # Return raw stats if requested
    return stats_df if return_data else None


I created the function and give to AI to boost it with comments + add rich for the visualisation

In [29]:
path_file = "/Volumes/SSD 1/Github/Dirty_Data_Sample_Kaggle/data/Dirty Data Sample.xlsx"
df = pd.read_excel(path_file) # import data

In [ ]:
data_overview(df) # use the function of visualisation

                             Data Overview Menu                             
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No. ┃ Option                        ┃ Description                        ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1  │ Missing values                │ Show missing values                │
│  2  │ Unique values                 │ Show unique values                 │
│  3  │ DataFrame head (preview)      │ Show dataframe head (preview)      │
│  4  │ Descriptive statistics        │ Show descriptive statistics        │
│  5  │ Info (non-null, type, memory) │ Show info (non-null, type, memory) │
│  6  │ Duplicates                    │ Show duplicates                    │
│  7  │ All                           │ Show all                           │
│  8  │ Quit                          │ Show quit                          │
└─────┴───────────────────────────────┴────────────────────────────────────┘

Enter choice(s) separated by comma: 

All information:

Shape: 825 rows, 16 columns (Memory: 0.38 MB)

Missing values:

...

           Duplicates           
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Total rows ┃ Duplicated rows ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│        825 │               0 │
└────────────┴─────────────────┘

Done. Choose another option or 8 to quit

                             Data Overview Menu                             
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No. ┃ Option                        ┃ Description                        ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1  │ Missing values                │ Show missing values                │
│  2  │ Unique values                 │ Show unique values                 │
│  3  │ DataFrame head (preview)      │ Show dataframe head (preview)      │
│  4  │ Descriptive statistics        │ Show descriptive statistics        │
│  5  │ Info (non-null, type, memory) │ Show info (non-null, type, memory) │
│  6  │ Duplicates                    │ Show duplicates                    │
│  7  │ All                           │ Show all                           │
│  8  │ Quit                          │ Show quit                          │
└─────┴───────────────────────────────┴────────────────────────────────────┘

Enter choice(s) separated by comma: 

Exiting menu...

As we can see, with the overview, our data are pivot, we will need to unpivot them to have access to the real data. 

That's the main reason why we have so much NaN here.

In [32]:
# Removed the columns total with have all the other columns so we don't need this one
df.drop(columns = ['Consumer Total','Corporate Total','Home Office Total'],axis = 1,inplace = True)

In [33]:
# Create empty lists to store results
segments, ship_modes, order_ids, sales = [], [], [], []

# Define mapping between column index ranges and segment names
segment_map = {
    range(1, 5): "Consumer",
    range(5, 9): "Corporate",
    range(9, 13): "Home Office"
}

# Loop through the relevant columns (1 to 12 included)
for i in range(1, 13):
    col_values = df.iloc[:, i].to_numpy()
    ship_mode = col_values[0]  # The first row contains the Ship Mode
    
    # Identify the right segment name from the mapping
    segment_name = next(seg for rng, seg in segment_map.items() if i in rng)
    
    # Loop through rows 2 → 824 (not including headers or totals)
    for row_idx in range(2, 824):  
        value = df.iat[row_idx, i]
        if pd.notna(value):  # Skip NaN
            segments.append(segment_name)
            ship_modes.append(ship_mode)
            order_ids.append(df.iat[row_idx, 0])  # First column = OrderID
            sales.append(value)
    
    # Add the Grand Total row 
    segments.append(segment_name)
    ship_modes.append(ship_mode)
    order_ids.append("Grand Total")
    sales.append(df.iat[824, i])

# Build the tidy DataFrame
df_final = pd.DataFrame({
    "Segment": segments,
    "Ship Mode": ship_modes,
    "OrderID": order_ids,
    "Sales": sales
})


Instead of repeating the same if blocks three times, I use a dictionary (segment_map) to decide whether a column belongs to "Consumer", "Corporate", or "Home Office".

→ This avoids writing the same condition three times.

I use __.iat__ instead of __.iloc[m, i]__ inside the loop because it’s faster for single-cell access.

__The row indices are more explicit:__

- 2 → 824 for data rows
-  825 for the Grand Total row (the request).
- At the end, I gather everything in a clean DataFrame.

In [35]:
data_overview(df_final)

                             Data Overview Menu                             
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No. ┃ Option                        ┃ Description                        ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1  │ Missing values                │ Show missing values                │
│  2  │ Unique values                 │ Show unique values                 │
│  3  │ DataFrame head (preview)      │ Show dataframe head (preview)      │
│  4  │ Descriptive statistics        │ Show descriptive statistics        │
│  5  │ Info (non-null, type, memory) │ Show info (non-null, type, memory) │
│  6  │ Duplicates                    │ Show duplicates                    │
│  7  │ All                           │ Show all                           │
│  8  │ Quit                          │ Show quit                          │
└─────┴───────────────────────────────┴────────────────────────────────────┘

Enter choice(s) separated by comma: 

DataFrame head (preview):

┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Index ┃ Segment  ┃ Ship Mode   ┃ OrderID        ┃ Sales              ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│     0 │ Consumer │ First Class │ CA-2011-103366 │ 149.95             │
│     1 │ Consumer │ First Class │ CA-2011-109043 │ 243.60000000000002 │
│     2 │ Consumer │ First Class │ CA-2011-113166 │ 9.568000000000001  │
│     3 │ Consumer │ First Class │ CA-2011-124023 │ 8.96               │
│     4 │ Consumer │ First Class │ CA-2011-130155 │ 34.2               │
└───────┴──────────┴─────────────┴────────────────┴────────────────────┘

DataFrame Info:

                 DataFrame Info                 
┏━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Column    ┃ Non-null ┃ Type    ┃ Memory (MB) ┃
┡━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Segment   │      834 │ object  │        0.05 │
├───────────┼──────────┼─────────┼─────────────┤
│ Ship Mode │      834 │ object  │        0.06 │
├───────────┼──────────┼─────────┼─────────────┤
│ OrderID   │      834 │ object  │        0.06 │
├───────────┼──────────┼─────────┼─────────────┤
│ Sales     │      834 │ float64 │        0.01 │
└───────────┴──────────┴─────────┴─────────────┘

Done. Choose another option or 8 to quit

                             Data Overview Menu                             
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No. ┃ Option                        ┃ Description                        ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1  │ Missing values                │ Show missing values                │
│  2  │ Unique values                 │ Show unique values                 │
│  3  │ DataFrame head (preview)      │ Show dataframe head (preview)      │
│  4  │ Descriptive statistics        │ Show descriptive statistics        │
│  5  │ Info (non-null, type, memory) │ Show info (non-null, type, memory) │
│  6  │ Duplicates                    │ Show duplicates                    │
│  7  │ All                           │ Show all                           │
│  8  │ Quit                          │ Show quit                          │
└─────┴───────────────────────────────┴────────────────────────────────────┘

Enter choice(s) separated by comma: 

Exiting menu...

__Excel (Power Query + DAX):__

it’s the quick and easy way. You just click Unpivot, then write a couple of formulas. Perfect for everyday analysis because it’s visual and fast.

__Python:__

it’s a bit more technical, but very powerful. You write some code to do the same thing. It’s better if you need to repeat the process on lots of files, deal with huge data, or if you want to show recruiters you can automate things.

👉 So basically:

- Use Excel when you want speed and simplicity.

- Use Python when you need automation, big data handling, or to look pro in data projects.